# ChuckleNet WavLM — Precision Fix via Threshold Sweep

**Goal:** Fix precision gap by finding optimal decision threshold.

| Config | Threshold | F1 | Precision | Recall |
|--------|-----------|----|-----------|--------|
| Baseline | 0.50 | 0.6173 | 0.464 | 0.921 |
| Optimal | ? | ? | ? | ? |

In [ ]:
# 0. DIAGNOSTIC: Find where chuckle_checkpoints is mounted
import os

print('=== GDrive Mount Diagnostic ===')

# Check standard paths
for base in ['/content/drive/MyDrive', '/content/gdrive/MyDrive',
             '/content/drive/Shareddrives', '/gdrive/MyDrive']:
    if os.path.exists(base):
        items = sorted(os.listdir(base))
        print(f'\n{base}:')
        for item in items:
            if 'chuckle' in item.lower() or 'audio' in item.lower():
                print(f'  >>> {item} <<<')
            else:
                print(f'  {item}')
    else:
        print(f'{base}: does NOT exist')

print('\n=== Contents of /content/ ===')
for item in sorted(os.listdir('/content/')):
    print(f'  {item}')

In [ ]:
# 1. SETUP PATHS — edit these if the diagnostic above shows a different path
# Standard path for most users:
BASE = '/content/drive/MyDrive'
OUTPUT_DIR = f'{BASE}/chuckle_checkpoints'
AUDIO_DIR = f'{OUTPUT_DIR}/audio'
SEGMENTS_PATH = f'{OUTPUT_DIR}/aligned_utterances.jsonl'

# If diagnostic shows a different mount (e.g. Shared Drive), edit here:
# BASE = '/content/drive/Shareddrives/Your Shared Drive Name'
# OUTPUT_DIR = f'{BASE}/chuckle_checkpoints'

import os
for p in [SEGMENTS_PATH, AUDIO_DIR]:
    status = 'OK' if os.path.exists(p) else 'MISSING'
    print(f'  {status}: {p}')

n_audio = len([f for f in os.listdir(AUDIO_DIR) if f.endswith('.mp3')]) if os.path.exists(AUDIO_DIR) else 0
print(f'  Audio files: {n_audio}')

SR = 16000
MAX_LEN = int(5.0 * SR)

In [ ]:
# 2. Setup + Install
!pip install -q transformers librosa

import torch, numpy as np, json, os, time, librosa
from collections import Counter
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# 3. Load model
print('Loading WavLM model...')
encoder = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
encoder.to(device)
encoder.eval()
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')

class WavLMLaughterClassifier(torch.nn.Module):
    def __init__(self, enc):
        super().__init__()
        self.encoder = enc
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(768, 256), torch.nn.ReLU(), torch.nn.Dropout(0.25),
            torch.nn.Linear(256, 64), torch.nn.ReLU(), torch.nn.Dropout(0.25),
            torch.nn.Linear(64, 2)
        )
    def forward(self, waveforms):
        with torch.no_grad():
            wav_list = [w.cpu().numpy() for w in waveforms]
            inputs = feature_extractor(wav_list, sampling_rate=SR, return_tensors='pt', padding=True)
            inputs = {k: v.to(waveforms.device) for k, v in inputs.items()}
            hidden = self.encoder(**inputs).last_hidden_state.mean(dim=1)
        return self.classifier(hidden)

model = WavLMLaughterClassifier(encoder).to(device)
ckpt_path = f'{OUTPUT_DIR}/wavlm_phaseA_best.pt'
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Model loaded (epoch {ckpt['epoch']}, val_f1={ckpt['val_f1']:.4f})")

In [ ]:
# 4. Load test data (80/20 video split — identical to training)
print(f'Loading {SEGMENTS_PATH}...')
samples = []
with open(SEGMENTS_PATH) as f:
    for line in f:
        d = json.loads(line)
        vid = d.get('video_id', '')
        audio_path = f'{AUDIO_DIR}/{vid}.mp3'
        if not os.path.exists(audio_path):
            continue
        label = d.get('label_any', 0)
        start, end = d.get('start'), d.get('end')
        if start is not None and end is not None and end > start:
            samples.append({
                'audio_path': audio_path,
                'start': start, 'end': end,
                'label': int(label), 'video_id': vid
            })

print(f'Loaded {len(samples)} segments')

# 80/20 video-level split (same seed=42 as training)
all_vids = list(set(s['video_id'] for s in samples))
np.random.seed(42)
np.random.shuffle(all_vids)
split_idx = int(0.8 * len(all_vids))
test_vids = set(all_vids[split_idx:])
test_samples = [s for s in samples if s['video_id'] in test_vids]
print(f'Test: {len(test_samples)} samples from {len(test_vids)} held-out videos')

In [ ]:
# 5. Collect prediction probabilities (all test samples)
print('Running inference on all test samples...')
t0 = time.time()
all_probs, all_labels = [], []

with torch.no_grad():
    for i, s in enumerate(test_samples):
        if i % 500 == 0:
            print(f'  [{i}/{len(test_samples)}]')
        try:
            va, _ = librosa.load(s['audio_path'], sr=SR, mono=True,
                                offset=max(0, s['start']-0.05), duration=5.2)
        except:
            va = np.zeros(MAX_LEN, dtype=np.float32)
        if va.shape[0] > MAX_LEN:
            va = va[:MAX_LEN]
        else:
            va = np.pad(va, (0, MAX_LEN - va.shape[0]))
        vx = torch.from_numpy(va[None]).to(device)
        prob = torch.softmax(model(vx), dim=1)[:, 1].cpu().item()
        all_probs.append(prob)
        all_labels.append(s['label'])

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
print(f'Done in {time.time()-t0:.0f}s')
print(f'Prob: min={all_probs.min():.3f} max={all_probs.max():.3f} mean={all_probs.mean():.3f}')
print(f'Labels: {dict(Counter(all_labels))}')

In [ ]:
# 6. Threshold sweep
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'TP':>6} {'FP':>6} {'FN':>6}")
print('-' * 70)

best_f1, best_t = 0, 0.5
results = []

for t in [0.30, 0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    preds = (all_probs >= t).astype(int)
    tp = int(((preds == 1) & (all_labels == 1)).sum())
    fp = int(((preds == 1) & (all_labels == 0)).sum())
    fn = int(((preds == 0) & (all_labels == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f'{t:>10.2f} {precision:>10.4f} {recall:>10.4f} {f1:>10.4f} {tp:>6} {fp:>6} {fn:>6}')
    results.append({'threshold': t, 'precision': precision, 'recall': recall,
                    'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn})
    if f1 > best_f1:
        best_f1, best_t = f1, t

print()
print('=' * 70)
improvement = best_f1 - 0.6173
print(f'BEST:  t={best_t:.2f}  F1={best_f1:.4f}  P={results[[r["threshold"] for r in results].index(best_t)]["precision"]:.4f}  R={results[[r["threshold"] for r in results].index(best_t)]["recall"]:.4f}')
print(f'       vs baseline F1=0.6173  →  +{improvement:.4f} ({100*improvement/0.6173:.1f}% relative)')
print()
print('=== Publication-quality candidates (P ≥ 0.55, R ≥ 0.70) ===')
for r in results:
    if r['precision'] >= 0.55 and r['recall'] >= 0.70:
        marker = ' ← BEST' if r['threshold'] == best_t else ''
        print(f"  t={r['threshold']:.2f}: F1={r['f1']:.4f} P={r['precision']:.4f} R={r['recall']:.4f}{marker}")

In [ ]:
# 7. Visualization
import matplotlib.pyplot as plt

thresholds  = [r['threshold'] for r in results]
precisions  = [r['precision'] for r in results]
recalls     = [r['recall'] for r in results]
f1s         = [r['f1'] for r in results]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, precisions, 'b-o', label='Precision', linewidth=2, markersize=6)
ax.plot(thresholds, recalls,    'g-o', label='Recall',    linewidth=2, markersize=6)
ax.plot(thresholds, f1s,        'r-s', label='F1 Score',  linewidth=2, markersize=6)
ax.axvline(x=best_t, color='red', linestyle='--', alpha=0.8, label=f'Best t={best_t}')
ax.axvline(x=0.5, color='orange', linestyle=':', alpha=0.8, label='Default t=0.5')
ax.axhline(y=0.6173, color='gray', linestyle='-.', alpha=0.5, label='Baseline F1=0.6173')
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('ChuckleNet WavLM — Threshold Sweep (Test Set, N=3341)', fontsize=13)
ax.legend(loc='best')
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/threshold_sweep.png', dpi=120)
plt.show()
print(f'Saved: {OUTPUT_DIR}/threshold_sweep.png')